# Pipeline 4: Academic Struggle Risk — Trauma & Assault Pattern Analysis

## 1. Problem Framing

**Business Question:** Which residents are at risk of struggling academically, and what trauma and assault patterns most predict that risk?

**Stakeholder:** Social workers managing education plans and case assignments.

**Why it matters:** Residents at Lighthouse Sanctuary have experienced trafficking, physical abuse, sexual abuse, and other trauma. These experiences can manifest as behavioral incidents, self-harm, poor emotional regulation, and disengagement from education. Early identification allows case workers to proactively intensify counseling, education support, and intervention plans before academic outcomes deteriorate.

**Ethical Disclaimer:** This model is a decision-support tool only. No automated action should be taken based on the risk score alone. All outputs require social worker review and clinical judgment. This tool must not be used to restrict services, limit education access, or make placement decisions without a full social worker assessment.

**Population Note:** All 60 residents are female. This analysis is specific to this population and should not be generalized to other demographics without revalidation.

**Dual Modeling Goals:**
- **Explanatory (Logistic Regression):** Understand *which* trauma and behavioral patterns statistically co-occur with school struggle (odds ratios, Ch. 9–11, 13)
- **Predictive (Gradient Boosting):** Score each resident on their risk of school struggle for case prioritization (Ch. 14–16)

## 2. Data Acquisition, Preparation & Exploration

In [ ]:
import sys
sys.path.insert(0, '..')

from pyLibrary import (
    univariate, unistats, bivariate, correlation_heatmap,
    missing_data_diagnostics, missing_data_clean, basic_wrangling,
    transform_skew, cap_outliers_iqr,
    build_preprocessor, make_pipeline_for_model, split_data,
    eval_classification, plot_roc_curve, plot_confusion_matrix,
    cross_validate_model, plot_learning_curve,
    tune_grid, select_features_rfe,
    permutation_importance_report, feature_importance_plot,
    plot_logit_coefficients, ols_summary,
    compute_vif, remove_high_vif,
    save_model, save_metrics
)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import LeaveOneOut, cross_val_score, StratifiedKFold
import warnings
warnings.filterwarnings('ignore')

DATA_DIR     = Path('../data/lighthouse_csv_v7')
RANDOM_STATE = 42
TARGET       = 'school_struggle'

In [ ]:
# ── Load all related tables ─────────────────────────────────────────────────
residents  = pd.read_csv(DATA_DIR / 'residents.csv',
                         parse_dates=['date_of_admission', 'date_enrolled', 'date_closed'])
education  = pd.read_csv(DATA_DIR / 'education_records.csv',        parse_dates=['record_date'])
health     = pd.read_csv(DATA_DIR / 'health_wellbeing_records.csv', parse_dates=['record_date'])
recordings = pd.read_csv(DATA_DIR / 'process_recordings.csv',       parse_dates=['session_date'])
incidents  = pd.read_csv(DATA_DIR / 'incident_reports.csv',         parse_dates=['incident_date'])
plans      = pd.read_csv(DATA_DIR / 'intervention_plans.csv',       parse_dates=['target_date'])

for name, tbl in [('residents',  residents),  ('education', education),
                   ('health',     health),     ('recordings', recordings),
                   ('incidents',  incidents),  ('plans',      plans)]:
    print(f'{name:12s}: {tbl.shape}')

print(f'\nAll residents are female: {(residents["sex"] == "F").all()}')
print(f'Incident types: {incidents["incident_type"].value_counts().to_dict()}')

### 2a. Target Variable: School Struggle Definition

The target variable must capture "struggling in school" in a way that is clinically meaningful, balanced enough for classification, and free of leakage.

**Chosen Definition — OR-criterion binary flag:**
- `school_struggle = 1` if `mean(attendance_rate) < 0.75` **OR** `mean(progress_percent) < 60`

**Rationale:**
- The 0.75 attendance threshold corresponds to chronic absenteeism (missing more than 1 in 4 school days), a well-established educational risk indicator.
- The 60% progress threshold marks borderline/failing academic performance.
- The OR logic yields ~57% positive cases — near-balanced classes, essential for small N classification.

**Leakage prevention:** `mean_attendance` and `mean_progress` define the target and are therefore **excluded** from the feature matrix. Only trend, recency, and completion-derived education features are used as predictors.

**Alternative target definitions considered (and why not chosen):**
| Alternative | Description | Issue |
|---|---|---|
| Trend-based | Flag declining 3-month attendance trend | Drops residents with fewer than 4 records; noisy at small window |
| Composite regression | Continuous score: mean_att + mean_prog/100 | R² meaningless at N=60; harder to communicate to caseworkers |
| Plan-gap | Attendance below Education plan target (0.85) | Flags 59/60 residents — zero discriminative power |

In [ ]:
# ── Target variable: school struggle (Ch. 11 — define target before features) ─

REFERENCE_DATE = pd.Timestamp('today').normalize()

edu_agg = education.groupby('resident_id').agg(
    mean_attendance   = ('attendance_rate',  'mean'),
    mean_progress     = ('progress_percent', 'mean'),
    edu_record_count  = ('record_date',      'count'),
    completion_rate   = ('completion_status', lambda x: (x == 'Completed').mean()),
    not_started_rate  = ('completion_status', lambda x: (x == 'NotStarted').mean()),
    # Recency proxy — most recent record values
    last_attendance   = ('attendance_rate',  'last'),
    last_progress     = ('progress_percent', 'last'),
    # Trend: last minus first (positive = improving, negative = declining)
    attendance_trend  = ('attendance_rate',
                         lambda x: float(x.iloc[-1] - x.iloc[0]) if len(x) > 1 else 0.0),
    progress_trend    = ('progress_percent',
                         lambda x: float(x.iloc[-1] - x.iloc[0]) if len(x) > 1 else 0.0),
).reset_index()

# Binary target: struggling in school
edu_agg[TARGET] = (
    (edu_agg['mean_attendance'] < 0.75) | (edu_agg['mean_progress'] < 60)
).astype(int)

print('Target distribution:')
print(edu_agg[TARGET].value_counts())
print(f'Positive rate: {edu_agg[TARGET].mean():.1%}')

### 2b. Feature Engineering

Features are aggregated to one row per resident across five tables. **Leakage rule:** `mean_attendance` and `mean_progress` are components of the target and must not appear in the feature matrix.

In [ ]:
# ── 3A: Assault and trauma features from residents.csv ──────────────────────
# Primary assault signals: sub-category abuse flags (stored as 'True'/'False' strings)

ABUSE_COLS = ['sub_cat_physical_abuse', 'sub_cat_sexual_abuse',
              'sub_cat_trafficked', 'sub_cat_osaec', 'sub_cat_child_labor',
              'sub_cat_at_risk', 'sub_cat_street_child']

assault_df = residents[['resident_id'] + ABUSE_COLS].copy()
for col in ABUSE_COLS:
    assault_df[col] = (assault_df[col].astype(str).str.lower() == 'true').astype(int)

assault_df['trauma_subcat_count'] = assault_df[ABUSE_COLS].sum(axis=1)
assault_df['has_sexual_abuse']    = assault_df['sub_cat_sexual_abuse']
assault_df['has_physical_abuse']  = assault_df['sub_cat_physical_abuse']
assault_df['has_trafficking']     = (
    (assault_df['sub_cat_trafficked'] == 1) | (assault_df['sub_cat_osaec'] == 1)
).astype(int)

print('Trauma subcat distribution:')
print(assault_df['trauma_subcat_count'].value_counts().sort_index())
print(f'\nResidents with \u22651 abuse sub-category: {(assault_df["trauma_subcat_count"] > 0).sum()}')

In [ ]:
# ── 3B: Incident features (behavioral trauma proxies) ────────────────────────
# Note: no "Assault" incident type exists. Trauma-response incident proxies are:
#   SelfHarm, RunawayAttempt, Security

sev_map = {'Low': 1, 'Medium': 2, 'High': 3}
incidents['severity_num'] = incidents['severity'].map(sev_map)

inc_agg = incidents.groupby('resident_id').agg(
    incident_count           = ('incident_id',      'count'),
    self_harm_count          = ('incident_type',    lambda x: (x == 'SelfHarm').sum()),
    runaway_attempt_count    = ('incident_type',    lambda x: (x == 'RunawayAttempt').sum()),
    security_count           = ('incident_type',    lambda x: (x == 'Security').sum()),
    behavioral_count         = ('incident_type',    lambda x: (x == 'Behavioral').sum()),
    conflict_count           = ('incident_type',    lambda x: (x == 'ConflictWithPeer').sum()),
    high_severity_count      = ('severity',         lambda x: (x == 'High').sum()),
    mean_severity            = ('severity_num',     'mean'),
    unresolved_count         = ('resolved',         lambda x: (x.astype(str).str.lower() == 'false').sum()),
    follow_up_required_count = ('follow_up_required', lambda x: (x.astype(str).str.lower() == 'true').sum()),
    days_since_last_incident = ('incident_date',
                                lambda x: (REFERENCE_DATE - x.max()).days),
    incident_span_days       = ('incident_date',
                                lambda x: (x.max() - x.min()).days if len(x) > 1 else 0),
).reset_index()

# Derived features
inc_agg['trauma_incident_count'] = (
    inc_agg['self_harm_count'] +
    inc_agg['runaway_attempt_count'] +
    inc_agg['security_count']
)
inc_agg['high_sev_rate']         = inc_agg['high_severity_count'] / inc_agg['incident_count'].clip(lower=1)
inc_agg['recent_incident_flag']  = (inc_agg['days_since_last_incident'] < 90).astype(int)

print('Incident aggregation shape:', inc_agg.shape)
print(f'Residents with incidents: {inc_agg["resident_id"].nunique()} / 60')

In [ ]:
# ── 3C: Resident background features ─────────────────────────────────────────

res_features = residents[['resident_id', 'initial_risk_level', 'current_risk_level',
                           'date_of_admission', 'age_upon_admission', 'has_special_needs',
                           'is_pwd', 'family_is_4ps', 'family_solo_parent',
                           'family_indigenous', 'family_parent_pwd',
                           'family_informal_settler', 'case_category']].copy()

# Age: parse "15 Years 9 months" → float years
res_features['age_years'] = (
    res_features['age_upon_admission']
    .str.extract(r'(\d+)\s*[Yy]ears?')[0]
    .astype(float)
)

# Risk level encoding
risk_map = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 3}
res_features['initial_risk_enc'] = res_features['initial_risk_level'].map(risk_map)
res_features['current_risk_enc'] = res_features['current_risk_level'].map(risk_map)
res_features['risk_improvement'] = res_features['initial_risk_enc'] - res_features['current_risk_enc']

# Length of stay
res_features['los_months'] = (
    (REFERENCE_DATE - res_features['date_of_admission']).dt.days / 30.44
)

# Boolean flags → int
for col in ['has_special_needs', 'is_pwd', 'family_is_4ps', 'family_solo_parent',
            'family_indigenous', 'family_parent_pwd', 'family_informal_settler']:
    res_features[col] = (res_features[col].astype(str).str.lower() == 'true').astype(int)

res_features['family_vulnerability_count'] = res_features[[
    'family_is_4ps', 'family_solo_parent', 'family_indigenous',
    'family_parent_pwd', 'family_informal_settler'
]].sum(axis=1)

print('Resident features shape:', res_features.shape)

In [ ]:
# ── 3D: Health and wellbeing features ────────────────────────────────────────

health_agg = health.groupby('resident_id').agg(
    avg_general_health  = ('general_health_score',      'mean'),
    avg_nutrition       = ('nutrition_score',            'mean'),
    avg_sleep           = ('sleep_quality_score',        'mean'),
    avg_energy          = ('energy_level_score',         'mean'),
    avg_bmi             = ('bmi',                        'mean'),
    health_record_count = ('record_date',                'count'),
    checkup_rate        = ('medical_checkup_done',
                           lambda x: x.astype(str).str.lower().eq('true').mean()),
    psych_checkup_rate  = ('psychological_checkup_done',
                           lambda x: x.astype(str).str.lower().eq('true').mean()),
    health_trend        = ('general_health_score',
                           lambda x: float(x.iloc[-1] - x.iloc[0]) if len(x) > 1 else 0.0),
).reset_index()

# Composite wellbeing index (all scores on same 1-5 scale)
health_agg['wellbeing_index'] = health_agg[[
    'avg_general_health', 'avg_nutrition', 'avg_sleep', 'avg_energy'
]].mean(axis=1)

print('Health aggregation shape:', health_agg.shape)

In [ ]:
# ── 3E: Counseling engagement features ───────────────────────────────────────

NEGATIVE_START = {'Angry', 'Distressed', 'Anxious', 'Sad', 'Withdrawn'}
NEGATIVE_END   = {'Sad', 'Anxious', 'Withdrawn', 'Distressed'}

recordings['concerns_bool']  = recordings['concerns_flagged'].astype(str).str.lower() == 'true'
recordings['progress_bool']  = recordings['progress_noted'].astype(str).str.lower() == 'true'
recordings['referral_bool']  = recordings['referral_made'].astype(str).str.lower() == 'true'
recordings['neg_start']      = recordings['emotional_state_observed'].isin(NEGATIVE_START).astype(int)
recordings['neg_end']        = recordings['emotional_state_end'].isin(NEGATIVE_END).astype(int)
recordings['emo_improvement'] = (recordings['neg_start'] - recordings['neg_end']).clip(lower=0)

rec_agg = recordings.groupby('resident_id').agg(
    session_count             = ('recording_id',          'count'),
    total_session_minutes     = ('session_duration_minutes', 'sum'),
    avg_session_minutes       = ('session_duration_minutes', 'mean'),
    individual_session_rate   = ('session_type',           lambda x: (x == 'Individual').mean()),
    concern_rate              = ('concerns_bool',          'mean'),
    progress_rate             = ('progress_bool',          'mean'),
    referral_rate             = ('referral_bool',          'mean'),
    negative_start_rate       = ('neg_start',              'mean'),
    negative_end_rate         = ('neg_end',                'mean'),
    emotional_improvement_rate = ('emo_improvement',       'mean'),
).reset_index()

print('Recordings aggregation shape:', rec_agg.shape)

In [ ]:
# ── 3F: Intervention plan features ───────────────────────────────────────────

plan_agg = plans.groupby('resident_id').agg(
    total_plan_count          = ('plan_id',       'count'),
    plan_completion_rate      = ('status',        lambda x: x.isin(['Achieved', 'Closed']).mean()),
    education_plan_active     = ('plan_category', lambda x: (
        (x == 'Education') & ~x.isin(['Achieved', 'Closed'])
    ).any()),
    education_plan_achieved   = ('plan_category', lambda x: (
        (x == 'Education') & (x == 'Achieved')
    ).any()),
).reset_index()

for col in ['education_plan_active', 'education_plan_achieved']:
    plan_agg[col] = plan_agg[col].astype(int)

print('Plans aggregation shape:', plan_agg.shape)

In [ ]:
# ── 3G: Join all features → final feature matrix ─────────────────────────────

# Start with resident-level assault features
df = res_features.merge(assault_df[['resident_id', 'trauma_subcat_count',
                                     'has_sexual_abuse', 'has_physical_abuse',
                                     'has_trafficking'] + ABUSE_COLS], on='resident_id', how='left')

# Merge aggregations
for agg_df in [edu_agg, health_agg, rec_agg, plan_agg]:
    df = df.merge(agg_df, on='resident_id', how='left')

# Left merge incidents (some residents have no incidents → zero-fill)
df = df.merge(inc_agg, on='resident_id', how='left')
INC_COLS = [c for c in inc_agg.columns if c != 'resident_id']
df[INC_COLS] = df[INC_COLS].fillna(0)

print('Final matrix shape:', df.shape)
print('\nMissing values:')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'None \u2014 clean!')

# ── CRITICAL: Leakage assertion ───────────────────────────────────────────────
# mean_attendance and mean_progress define the target; they must NOT be features
assert 'mean_attendance' not in df.drop(columns=[TARGET]).columns, "LEAKAGE: mean_attendance in features!"
assert 'mean_progress'   not in df.drop(columns=[TARGET]).columns, "LEAKAGE: mean_progress in features!"
print('\nLeakage check passed.')

## 3. Exploratory Data Analysis (EDA)

All EDA uses only the training distribution. We examine: distributions, feature-target relationships, multicollinearity, skew, and outliers before any modeling.

In [ ]:
# ── Define clean feature matrix (drop non-feature columns) ───────────────────

DROP_COLS = ['resident_id', 'initial_risk_level', 'current_risk_level',
             'date_of_admission', 'age_upon_admission',
             # These define the target — excluded to prevent leakage
             'mean_attendance', 'mean_progress']

feat_df = df.drop(columns=[c for c in DROP_COLS if c in df.columns], errors='ignore').copy()

# Keep case_category as a categorical predictor
print('Feature matrix shape:', feat_df.shape)
print('Target distribution:')
print(feat_df[TARGET].value_counts())
print(f'Positive rate (school struggle): {feat_df[TARGET].mean():.1%}')

In [ ]:
# ── Missing data diagnostics (Ch. 7) ─────────────────────────────────────────
missing_data_diagnostics(feat_df.drop(columns=[TARGET]))

In [ ]:
# ── Univariate stats (Ch. 6) ──────────────────────────────────────────────────
unistats(feat_df)

In [ ]:
# ── Bivariate analysis: every feature vs. target (Ch. 8) ─────────────────────
bivariate(feat_df, target=TARGET)

In [ ]:
# ── Correlation heatmap (Ch. 8) ───────────────────────────────────────────────
correlation_heatmap(feat_df)

In [ ]:
# ── Skew transformation and outlier capping (Ch. 7) ──────────────────────────
SKEW_COLS = ['incident_count', 'trauma_incident_count', 'self_harm_count',
             'runaway_attempt_count', 'security_count',
             'session_count', 'total_session_minutes', 'los_months',
             'days_since_last_incident', 'incident_span_days']

skew_cols_present = [c for c in SKEW_COLS if c in feat_df.columns]
feat_df = transform_skew(feat_df, features=skew_cols_present)
feat_df = cap_outliers_iqr(feat_df, cols=skew_cols_present)
print('Skew transformation and IQR capping applied to:', skew_cols_present)

## 4. Explanatory Model: Logistic Regression + Odds Ratios

**Goal:** Identify which trauma and behavioral patterns *co-occur* with academic struggle.

**Small-N caveat:** With ~60 observations, p-values are unreliable. We report odds ratios and interpret features with OR > 1.5 or OR < 0.67 and p < 0.25 as clinically noteworthy signals. McFadden R² is the primary fit statistic.

Using Chapters 9–11, 13 methodology.

In [ ]:
# ── VIF check for multicollinearity (Ch. 10) ─────────────────────────────────
num_cols_for_vif = feat_df.select_dtypes(include=[np.number]).drop(columns=[TARGET]).columns.tolist()
X_num = feat_df[num_cols_for_vif].fillna(feat_df[num_cols_for_vif].median())

vif_df = compute_vif(X_num)
print('Top VIF scores:')
print(vif_df.sort_values('VIF', ascending=False).head(15))

X_low_vif = remove_high_vif(X_num, threshold=10.0)
print(f'\nFeatures retained after VIF filter: {X_low_vif.shape[1]} / {X_num.shape[1]}')

In [ ]:
# ── Explanatory model: statsmodels Logit (Ch. 9–11) ──────────────────────────
y_exp = feat_df[TARGET].values
X_exp = sm.add_constant(X_low_vif.astype(float))

try:
    logit_res = sm.Logit(y_exp, X_exp).fit(disp=False, maxiter=200)
    print(logit_res.summary())
    mcfadden_r2 = 1 - logit_res.llf / logit_res.llnull
    print(f"\nMcFadden's R\u00b2: {mcfadden_r2:.4f}")
    convergence_ok = True
except Exception as e:
    print(f'Statsmodels Logit failed to converge: {e}')
    print('Falling back to sklearn LogisticRegression (L2 regularized).')
    convergence_ok = False

In [ ]:
# ── Odds ratios from explanatory logit ────────────────────────────────────────
if convergence_ok:
    odds_df = pd.DataFrame({
        'feature':    X_exp.columns,
        'coef':       logit_res.params,
        'odds_ratio': np.exp(logit_res.params),
        'ci_low':     np.exp(logit_res.conf_int()[0]),
        'ci_high':    np.exp(logit_res.conf_int()[1]),
        'p_value':    logit_res.pvalues
    }).sort_values('p_value').reset_index(drop=True)

    # Sanity check
    assert (odds_df['odds_ratio'] > 0).all(), "Sign error in odds ratios!"

    # Display top noteworthy features (p < 0.25, OR > 1.5 or < 0.67)
    noteworthy = odds_df[
        (odds_df['feature'] != 'const') & 
        (odds_df['p_value'] < 0.25) &
        ((odds_df['odds_ratio'] > 1.5) | (odds_df['odds_ratio'] < 0.67))
    ]
    print('Noteworthy features (OR > 1.5 or < 0.67, p < 0.25):')
    print(noteworthy[['feature','odds_ratio','ci_low','ci_high','p_value']].to_string())

In [ ]:
# ── Visualize odds ratios (Ch. 13) ───────────────────────────────────────────
if convergence_ok:
    plot_df = odds_df[odds_df['feature'] != 'const'].sort_values('odds_ratio')
    
    fig, ax = plt.subplots(figsize=(10, max(6, len(plot_df) * 0.35)))
    colors = ['#e74c3c' if or_ > 1 else '#2ecc71' for or_ in plot_df['odds_ratio']]
    ax.barh(plot_df['feature'], plot_df['odds_ratio'], color=colors, alpha=0.7)
    ax.axvline(x=1, color='black', linestyle='--', linewidth=1.5, label='OR = 1 (no effect)')
    ax.set_xlabel('Odds Ratio', fontsize=12)
    ax.set_title('Odds Ratios \u2014 School Struggle Risk\n(Red = increases risk, Green = reduces risk)', fontsize=13)
    ax.legend()
    plt.tight_layout()
    plt.show()
    
    print('\nInterpretation note: With N~60, p-values have wide CIs.')
    print('OR > 1.5 or < 0.67 with p < 0.25 are considered clinically noteworthy.')

## 5. Predictive Model: Gradient Boosting Classifier

**Goal:** Produce a calibrated risk score for each resident for case prioritization.

**Small-N strategy:**
- **LeaveOneOut (LOO) CV** is used as the primary validation metric — unbiased at any N
- **Shallow trees** (max_depth=2) and **subsampling** (subsample=0.8) prevent overfitting
- **Stratified train/test split** preserves class balance in both sets

**Primary metric:** LOO-CV AUC-ROC (threshold-independent, appropriate for the near-balanced target)

Using Chapters 14–16 methodology.

In [ ]:
# ── Train/test split (Ch. 11, 15) ────────────────────────────────────────────
X_train, X_test, y_train, y_test = split_data(
    feat_df, target=TARGET,
    test_size=0.2, random_state=RANDOM_STATE, stratify=True
)
print(f'Train: {len(X_train)} rows | Test: {len(X_test)} rows')
print(f'Class balance \u2014 Train: {y_train.mean():.1%} positive | Test: {y_test.mean():.1%} positive')
print('\nNote: Test set has only ~12 rows. LOO-CV is the primary evaluation metric.')

# ── Build leakage-free pipelines (Ch. 11) ─────────────────────────────────────
lr_pipe = make_pipeline_for_model(
    X_train,
    LogisticRegression(C=0.1, class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE)
)
gb_pipe = make_pipeline_for_model(
    X_train,
    GradientBoostingClassifier(
        n_estimators=100, max_depth=2, learning_rate=0.05,
        subsample=0.8, random_state=RANDOM_STATE
    )
)

In [ ]:
# ── Primary CV: LeaveOneOut — unbiased at small N (Ch. 15) ───────────────────
print('Running LeaveOneOut CV (this is the headline metric)...')

X_all = feat_df.drop(columns=[TARGET])
y_all = feat_df[TARGET]

loo = LeaveOneOut()
loo_auc_gb = cross_val_score(gb_pipe, X_all, y_all, cv=loo, scoring='roc_auc', n_jobs=-1)
loo_auc_lr = cross_val_score(lr_pipe, X_all, y_all, cv=loo, scoring='roc_auc', n_jobs=-1)

print(f'\n{"Model":<40} {"LOO-CV AUC":>12} {"Std":>8}')
print('-' * 62)
print(f'{"Gradient Boosting (predictive)":<40} {loo_auc_gb.mean():>12.4f} {loo_auc_gb.std():>8.4f}')
print(f'{"Logistic Regression (explanatory)":<40} {loo_auc_lr.mean():>12.4f} {loo_auc_lr.std():>8.4f}')
print('\nNote: LOO-CV std reflects sample-to-sample variability, not model instability.')

In [ ]:
# ── 5-fold stratified CV for comparison (Ch. 15) ─────────────────────────────
print('=== 5-Fold CV \u2014 AUC-ROC ===\n')
for name, pipe in [('Logistic Regression (explanatory)', lr_pipe),
                   ('Gradient Boosting (predictive)',     gb_pipe)]:
    print(f'{name}:')
    cross_validate_model(pipe, X_train, y_train, cv=5, scoring='roc_auc', stratified=True)
    print()

In [ ]:
# ── Learning curve: bias/variance diagnosis (Ch. 15) ──────────────────────────
plot_learning_curve(
    gb_pipe, X_all, y_all,
    cv=5, scoring='roc_auc',
    title='Learning Curve \u2014 Gradient Boosting (School Struggle Risk)'
)
print('High variance between train/CV curves is expected at N=60.')

In [ ]:
# ── Hyperparameter tuning (Ch. 15) ────────────────────────────────────────────
gb_param_grid = {
    'model__n_estimators':  [50, 100],
    'model__max_depth':     [2, 3],
    'model__learning_rate': [0.05, 0.10],
    'model__subsample':     [0.7, 0.9],
}
lr_param_grid = {
    'model__C': [0.01, 0.05, 0.1, 0.5]
}

print('Tuning Gradient Boosting...')
best_gb, gs_gb = tune_grid(gb_pipe, gb_param_grid, X_train, y_train, cv=5, scoring='roc_auc')
print(f'Best GBM params: {gs_gb.best_params_}  |  CV AUC: {gs_gb.best_score_:.4f}')

print('\nTuning Logistic Regression...')
best_lr, gs_lr = tune_grid(lr_pipe, lr_param_grid, X_train, y_train, cv=5, scoring='roc_auc')
print(f'Best LR params:  {gs_lr.best_params_}  |  CV AUC: {gs_lr.best_score_:.4f}')

## 6. Feature Selection (Ch. 14–16)

Feature selection at small N is critical to prevent overfitting. We use:
1. **RFECV** with Logistic Regression as base estimator (more stable than GBM at N=60)
2. **MDI feature importance** from the tuned GBM  
3. **Permutation importance** on the test set (unbiased, n_repeats=20)

In [ ]:
# ── Build preprocessor and transform data (Ch. 11) ────────────────────────────
preprocessor, num_cols_out, cat_cols_out = build_preprocessor(X_train)
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep  = preprocessor.transform(X_test)

# Get feature names after preprocessing
try:
    ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
    cat_feature_names = list(ohe.get_feature_names_out(cat_cols_out)) if cat_cols_out else []
except (KeyError, AttributeError):
    cat_feature_names = []

feature_names_out = num_cols_out + cat_feature_names
print(f'Preprocessed feature count: {len(feature_names_out)}')

# ── RFECV feature selection (Ch. 16) ──────────────────────────────────────────
print('\nRunning RFECV...')
_, selected_features = select_features_rfe(
    X_train_prep, y_train, feature_names_out,
    estimator=LogisticRegression(C=0.1, max_iter=1000, random_state=RANDOM_STATE),
    cv=5
)
print(f'Selected features ({len(selected_features)}): {selected_features}')

In [ ]:
# ── MDI feature importance from GBM (Ch. 14, 16) ─────────────────────────────
best_gb.fit(X_train, y_train)
feature_importance_plot(
    best_gb.named_steps['model'],
    feature_names_out,
    top_n=15,
    title='MDI Feature Importance \u2014 School Struggle Risk (GBM)'
)

# ── Permutation importance on TEST data (unbiased, Ch. 16) ───────────────────
pfi = permutation_importance_report(
    best_gb, X_test_prep, y_test,
    feature_names_out,
    n_repeats=20,
    scoring='roc_auc',
    top_n=15
)

In [ ]:
# ── Logistic regression coefficient plot (Ch. 13) ─────────────────────────────
best_lr.fit(X_train, y_train)
plot_logit_coefficients(
    best_lr,
    top_n=15,
    title='Logit Coefficients \u2014 School Struggle Risk (L2 Regularized)'
)

## 7. Evaluation & Interpretation

**Primary metric:** LOO-CV AUC-ROC (headline number)  
**Secondary metrics:** Test-set AUC, F1, Accuracy, Confusion Matrix  

**Honest uncertainty note:** At N=60, AUC confidence intervals are approximately ±0.10–0.15. The model's primary value is identifying *which features* co-occur with school struggle — not precise probability estimates. Social workers should treat the score as a conversation-starter, not a verdict.

In [ ]:
# ── Final evaluation on hold-out test set (Ch. 15) ────────────────────────────
print('=== Gradient Boosting (Predictive) ===')
results_gb = eval_classification(
    'Gradient Boosting (Tuned)', best_gb,
    X_train, y_train, X_test, y_test
)

print('\n=== Logistic Regression (Explanatory) ===')
results_lr = eval_classification(
    'Logistic Regression', best_lr,
    X_train, y_train, X_test, y_test, fit=False
)

In [ ]:
plot_roc_curve(
    best_gb, X_test, y_test,
    title='ROC Curve \u2014 School Struggle Risk (Gradient Boosting)'
)

plot_confusion_matrix(
    best_gb, X_test, y_test,
    title='Confusion Matrix \u2014 School Struggle Risk (Gradient Boosting)'
)

## 8. Risk Score Output

Risk tiers for operational use:
- **Lower Risk (0–33%)**: No immediate additional intervention needed
- **Moderate Risk (33–60%)**: Social worker check-in recommended; review education plan
- **High Risk (60–100%)**: Prioritize for immediate education support and counseling intensification

In [ ]:
# ── Risk scores for all 60 residents ─────────────────────────────────────────
# Refit on full dataset for final scores
X_all_features = feat_df.drop(columns=[TARGET])
y_all_labels   = feat_df[TARGET]
best_gb.fit(X_all_features, y_all_labels)

struggle_probs = best_gb.predict_proba(X_all_features)[:, 1]

score_df = df[['resident_id']].copy()
score_df['actual_school_struggle'] = y_all_labels.values
score_df['struggle_risk_score']    = (struggle_probs * 100).round(1)
score_df['struggle_risk_tier']     = pd.cut(
    struggle_probs,
    bins=[0, 0.33, 0.60, 1.0],
    labels=['Lower Risk', 'Moderate Risk', 'High Risk'],
    include_lowest=True
)

print('Risk tier distribution:')
print(score_df['struggle_risk_tier'].value_counts())

print('\nSample output (top 10 highest risk):')
print(score_df.sort_values('struggle_risk_score', ascending=False).head(10).to_string(index=False))

# Sanity check: output has exactly 60 rows
assert len(score_df) == 60, f"Expected 60 rows, got {len(score_df)}"
print(f'\nOutput verified: {len(score_df)} rows (all residents scored)')

score_df.to_csv('school_struggle_risk_scores.csv', index=False)
print('Scores saved: school_struggle_risk_scores.csv')

## 9. Alternative and Refined Model Approaches

The following approaches were considered and are recommended for future investigation:

### Alternative A: Rolling Window / Trend-Based Target
**Approach:** Flag residents whose last 3 months show declining attendance (`attendance_trend_3m < -0.05`).  
**Pros:** Forward-looking — catches deterioration before it becomes entrenched; more actionable.  
**Cons:** Drops residents with short stays (< 4 education records); high noise at small window size.  
**Recommendation:** Implement as a supplementary rule-based watch list using pandas rolling operations. No ML needed — simpler and more auditable.

### Alternative B: Continuous Composite Regression
**Approach:** Predict a continuous school score: `(mean_attendance + mean_progress/100 + completion_rate) / 3`. Use Ridge regression (explanatory) + GBM regressor (predictive).  
**Pros:** Preserves granularity in the outcome; simpler coefficient interpretation than log-odds.  
**Cons:** R² is unreliable at N=60; RMSE on a 0–1 scale is not interpretable as a standalone metric.  
**Recommendation:** Use during EDA as a continuous exploration tool, then threshold at the ≈25th percentile for the operational binary flag.

### Alternative C: Rule-Based Hybrid (Recommended for Production)
**Approach:**
- Tier 3 (High Risk — obvious): `mean_attendance < 0.65 AND (has_sexual_abuse OR has_trafficking OR self_harm_count > 0)`  
- Tier 1 (Lower Risk — obvious): `mean_attendance > 0.85 AND trauma_subcat_count == 0`  
- Tier 2 (Ambiguous): everyone else — run ML classifier here only

**Pros:** Fully auditable by social workers; no ML literacy required for the clear cases; ML adds value only where rules are insufficient.  
**Cons:** Rule thresholds need clinical expert validation before deployment.  
**Recommendation:** Deploy ML scores now, validate with social workers over 6 months, then codify agreed-upon rules as explicit Tier 3 / Tier 1 criteria.

## 10. Causal Limitations and Relationship Notes

**Observed correlations are NOT causal claims.** The features that predict school struggle in this model are associated with poor academic outcomes — but this dataset cannot support causal inference because:
- No control group (all residents have experienced trauma)
- No random assignment to interventions
- Confounding: residents with higher trauma burden also receive more intensive counseling (which may appear to *predict* struggle, but is actually a response to it)

**For causal analysis:** A difference-in-differences design comparing residents who received education-specific interventions vs. those who did not would require a larger dataset and longitudinal tracking beyond what is currently available.

**What this model IS useful for:** Triage and prioritization. Given two residents with similar backgrounds, the model helps surface which one deserves a proactive education check-in conversation first.

In [ ]:
# ── MLOps: save model and metrics (Ch. 17+) ──────────────────────────────────
final_metrics = {
    'model':             'GradientBoostingClassifier',
    'notebook':          'school-struggle-risk.ipynb',
    'target':            TARGET,
    'target_definition': 'mean_attendance < 0.75 OR mean_progress < 60',
    'n_total':           int(len(feat_df)),
    'n_train':           int(len(X_train)),
    'n_test':            int(len(X_test)),
    'positive_rate':     float(feat_df[TARGET].mean()),
    'loo_cv_auc_gb':     float(loo_auc_gb.mean()),
    'loo_cv_auc_gb_std': float(loo_auc_gb.std()),
    'loo_cv_auc_lr':     float(loo_auc_lr.mean()),
    'best_gb_params':    str(gs_gb.best_params_),
    'test_auc_gb':       float(results_gb.get('roc_auc', 0)),
    'test_f1_gb':        float(results_gb.get('f1', 0)),
    'disclaimer':        (
        'Small N (~60). Scores are directional indicators for case prioritization, '
        'not clinical diagnoses. Social worker review is required before any action.'
    )
}

save_metrics(final_metrics, 'school_struggle_metrics.json')
save_model(best_gb, 'school_struggle_model.sav')

print('Model saved:   school_struggle_model.sav')
print('Metrics saved: school_struggle_metrics.json')
print('Scores saved:  school_struggle_risk_scores.csv')
print('\n--- Final headline metrics ---')
print(f'LOO-CV AUC (GBM): {final_metrics["loo_cv_auc_gb"]:.4f} \u00b1 {final_metrics["loo_cv_auc_gb_std"]:.4f}')
print(f'Positive rate:    {final_metrics["positive_rate"]:.1%}')

## Deployment Notes

**Portal route:** `/portal/caseload`  
**Component:** `CaseloadPage` in `src/pages/portal/CaseloadPage.tsx`  
**Supabase table:** `public.resident_ml_scores` (columns: `school_struggle_band`, `school_struggle_score`)

### How scores reach the UI

1. Run this notebook to produce a final scored frame with `resident_id` and `school_struggle_prob`.

```python
def assign_risk_band(prob):
    if prob >= 0.60:
        return "High"
    elif prob >= 0.30:
        return "Medium"
    else:
        return "Low"

scored["school_struggle_band"] = scored["school_struggle_prob"].apply(assign_risk_band)
scored["school_struggle_score"] = scored["school_struggle_prob"]
```

2. Export to Supabase:

```python
from supabase import create_client
import os

client = create_client(os.environ["SUPABASE_URL"], os.environ["SUPABASE_SERVICE_KEY"])

rows = (
    scored[["resident_id", "school_struggle_band", "school_struggle_score"]]
    .assign(model_version="1.0")
    .to_dict(orient="records")
)
client.table("resident_ml_scores").upsert(rows, on_conflict="resident_id").execute()
print(f"Upserted {len(rows)} rows.")
```

### Where it appears in the app

- **Caseload table → School Risk column:** a chip showing High / Medium / Low per resident.
- **Resident detail modal → Model Insights section:** school struggle score and band.

### Decision recommendation

Residents with **High** school struggle risk should be flagged for additional educational support. Cross-reference with wellbeing scores — both dimensions declining simultaneously is a compounding risk signal.